In [1]:
import os
from dataset import load_dataset
from regression.GPR.gpr_model import GPRModel
from regression.GPR.gpr_sampling import lhs_nearest_sampling
import joblib

TRAIN_PATH_FULL="../../../../data/processed/full_train/train.csv"
TRAIN_PATH_SPLIT= "../../../../data/processed/split_85_15/train_85.csv"
TEST_PATH_SPLIT= "../../../../data/processed/split_85_15/test_15.csv"
OUTPUT_PATH= "../../../../output/GPR/model_target/"
MODEL_PATH="gpr_target.pkl"
TARGET_COL = "trq_target"
FEATURES=["trq_measured","mgt","ias","oat","np_ng_ratio","pa"]

In [2]:
train_df=load_dataset(TRAIN_PATH_SPLIT)
test_df=load_dataset(TEST_PATH_SPLIT)

In [3]:
# 2. Controllo esistenza modelli
if  os.path.exists(OUTPUT_PATH+MODEL_PATH):
    print("Modello pre-addestrato trovato! Caricamento in corso...")
    try:
        gpr_obj = joblib.load(OUTPUT_PATH+MODEL_PATH)
        print("SUCCESSO: Modello e Scaler caricati correttamente.")
        addestramento_necessario = False
    except Exception as e:
        print(f"Errore nel caricamento dei file (file corrotti?): {e}")
        addestramento_necessario = True
else:
    print("Modello non trovato o incompleto. Avvio procedura di addestramento...")
    addestramento_necessario = True

if addestramento_necessario:
    X_train_scaled, y_train, scaler = lhs_nearest_sampling(
        train_df, FEATURES, TARGET_COL, n_samples=5000
    )

    print("Addestramento GPR in corso (potrebbe richiedere qualche minuto)...")
    gpr_obj = GPRModel(random_state=42)
    gpr_obj.fit(X_train_scaled, y_train)

    try:
        joblib.dump(gpr_obj, OUTPUT_PATH+MODEL_PATH)
        print(f"SUCCESSO: Modello salvato in -> {OUTPUT_PATH+MODEL_PATH}")

    except Exception as e:
        print(f"ERRORE DURANTE IL SALVATAGGIO: {e}")

Modello pre-addestrato trovato! Caricamento in corso...
SUCCESSO: Modello e Scaler caricati correttamente.


In [4]:
test_X=scaler.fit_transform(test_df[FEATURES])
test_y=test_df[TARGET_COL]
gpr_obj.evaluate(test_X,test_y)

/Users/alessandropettinaro/PycharmProjects/PHM_2024_Project/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but GaussianProcessRegressor was fitted without feature names
  warnings.warn(


[46.27290439 71.3751694  81.97640114 ... 50.73088831 52.75532837
 54.11452127] [2053.08323712 1492.08680867 1796.57800491 ...  183.20297988 2180.85830105
 1885.58363888]
{'rmse': 1586.9446051333352, 'mae': 1441.3273255472122, 'r2': -12444.869759279673}
